# bigcompute.science Research Agent + GPU Compute

**You have a free T4 GPU right now** (if you enabled it). This notebook lets you:
1. **Compile & run** CUDA experiments on open math conjectures
2. **Review findings** with AI peer review (Gemini free, or OpenAI/Anthropic)
3. **Contribute results** back via PR

**Before you start:**
- **GPU**: Go to **Runtime → Change runtime type → T4 GPU** (free)
- **API key**: Get a free Gemini key at [aistudio.google.com/apikey](https://aistudio.google.com/apikey) (instant, no credit card)

Without the GPU runtime, you can still run reviews and analysis — just not CUDA experiments.

> Part of [bigcompute.science](https://bigcompute.science) — open computational mathematics.

In [ ]:
# === Step 1: Clone + install ===
import os
if not os.path.exists('/content/idontknow'):
    !git clone https://github.com/cahlen/idontknow.git /content/idontknow
else:
    !cd /content/idontknow && git pull
%cd /content/idontknow
!pip install -q httpx


In [ ]:
# === GPU Check ===
import subprocess
try:
    r = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                       capture_output=True, text=True, timeout=5)
    if r.returncode == 0 and r.stdout.strip():
        print(f'GPU: {r.stdout.strip()}')
        !nvidia-smi | head -8
    else:
        print('No GPU detected.')
        print('Go to Runtime -> Change runtime type -> T4 GPU (free)')
        print('You can still run reviews without a GPU.')
except FileNotFoundError:
    print('nvidia-smi not found.')
    print('Go to Runtime -> Change runtime type -> T4 GPU (free)')
    print('You can still run reviews without a GPU.')


In [ ]:
# === Step 2: Set your API key ===
# The agent tries providers in order: Anthropic -> OpenAI -> Gemini
# In Colab, Gemini is the easiest (free, you're already logged into Google)

import os

# --- Option A: Colab Secrets (recommended) ---
# Click the key icon in the sidebar, add GEMINI_API_KEY
try:
    from google.colab import userdata
    for key_name in ['GEMINI_API_KEY', 'GOOGLE_API_KEY', 'OPENAI_API_KEY', 'ANTHROPIC_API_KEY']:
        try:
            val = userdata.get(key_name)
            if val:
                os.environ[key_name] = val
                print(f'  Loaded {key_name} from Colab Secrets')
        except Exception:
            pass  # Secret not set, that's fine
except ImportError:
    pass  # Not in Colab

# --- Option B: Paste directly ---
# Get a free Gemini key: https://aistudio.google.com/apikey
# os.environ['GEMINI_API_KEY'] = 'AIza...'       # Gemini (free)
# os.environ['OPENAI_API_KEY'] = 'sk-proj-...'   # OpenAI
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...' # Anthropic

has = {k: bool(os.environ.get(k)) for k in ['GEMINI_API_KEY','GOOGLE_API_KEY','OPENAI_API_KEY','ANTHROPIC_API_KEY']}
available = [k.replace('_API_KEY','') for k,v in has.items() if v]
print(f'\nAvailable: {", ".join(available) if available else "NONE"}')
if not available:
    print('\nGet a free Gemini key: https://aistudio.google.com/apikey')
    print('Then paste above or add as Colab Secret')


In [ ]:
# === Step 3: Quick test — can the agent talk to an LLM? ===
import sys
sys.path.insert(0, '.')
from scripts.research_agent import call_llm

result = call_llm('Respond with exactly: {"status": "ok", "model": "your-name"}', 'test')
if result:
    print(f'LLM connection: OK')
    print(f'Response: {result[:100]}')
else:
    print('FAILED — check your API key above')

In [ ]:
# === Step 6: Dry run — see what the agent would do ===
import subprocess, os
result = subprocess.run(
    ['python3', 'scripts/research_agent.py', '--once', '--dry-run'],
    env={**os.environ},  # pass all env vars including API keys
    capture_output=False
)


In [ ]:
# === Step 7: Harvest results (see what experiments completed) ===
import subprocess, os
result = subprocess.run(
    ['python3', 'scripts/research_agent.py', '--once', '--phase', 'harvest'],
    env={**os.environ},
    capture_output=False
)


In [ ]:
# === Step 6: Compile CUDA kernels (auto-detects your GPU) ===
import subprocess

# Auto-detect GPU architecture
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,compute_cap', '--format=csv,noheader'],
                           capture_output=True, text=True)
    gpu_name, compute_cap = result.stdout.strip().split(', ')
    major, minor = compute_cap.split('.')
    sm = f'sm_{major}{minor}'
    print(f'Detected: {gpu_name} (compute {compute_cap} → {sm})')
except:
    sm = 'sm_75'  # fallback to T4
    print(f'GPU detection failed, using {sm} (T4 default)')

# Known GPU architectures:
# T4 = sm_75, A100 = sm_80, L4 = sm_89, RTX 4090 = sm_89
# H100 = sm_90, B200 = sm_100a, RTX 5090 = sm_120a

!nvidia-smi | head -5
print(f'\nCompiling with -arch={sm}...')

import os
kernels = [
    ('zaremba_density_gpu', 'scripts/experiments/zaremba-density/zaremba_density_gpu.cu', '-lm'),
    ('kronecker_gpu', 'scripts/experiments/kronecker-coefficients/kronecker_gpu.cu', '-lm'),
    ('ramanujan_gpu', 'scripts/experiments/ramanujan-machine/ramanujan_gpu.cu', '-lm'),
    ('class_v2', 'scripts/experiments/class-numbers/class_numbers_v2.cu', '-lpthread -lm'),
]

compiled = []
for binary, source, libs in kernels:
    if os.path.exists(source):
        ret = os.system(f'nvcc -O3 -arch={sm} -o {binary} {source} {libs} 2>&1')
        status = 'OK' if ret == 0 else 'FAILED'
        print(f'  {binary}: {status}')
        if ret == 0:
            compiled.append(binary)
    else:
        print(f'  {binary}: source not found')

print(f'\n{len(compiled)}/{len(kernels)} kernels compiled for {sm}.')


In [ ]:
# === Step 8: Quick Wins (ALL finish in seconds) ===

import os
os.makedirs('scripts/experiments/zaremba-density/results', exist_ok=True)

#──────────────────────────────────────────────────
# QUICK WIN 1: Verify Zaremba to 100,000 (< 2 sec)
#──────────────────────────────────────────────────
print('='*50)
print('Verifying Zaremba conjecture to 100,000...')
print('='*50)
!./zaremba_density_gpu 100000 1,2,3,4,5 | tee scripts/experiments/zaremba-density/results/gpu_A12345_1e5_colab.log

#──────────────────────────────────────────────────
# QUICK WIN 2: Find the 27 exceptions (< 2 sec)
#──────────────────────────────────────────────────
print('\n' + '='*50)
print('Finding Zaremba exceptions for A={1,2,3}...')
print('='*50)
!./zaremba_density_gpu 10000 1,2,3 | tee scripts/experiments/zaremba-density/results/gpu_A123_1e4_colab.log

#──────────────────────────────────────────────────
# QUICK WIN 3: Density with just {1,2} (< 1 sec)
#──────────────────────────────────────────────────
print('\n' + '='*50)
print('Density with just digits {1,2}...')
print('='*50)
!./zaremba_density_gpu 10000 1,2 | tee scripts/experiments/zaremba-density/results/gpu_A12_1e4_colab.log

print('\n' + '='*50)
print('ALL QUICK WINS COMPLETE!')
print('='*50)


In [ ]:
# === Your results! ===
import sys, os, glob, subprocess

# Make sure we're in the right directory
repo_dir = '/content/idontknow' if os.path.exists('/content/idontknow') else '.'
os.chdir(repo_dir)
sys.path.insert(0, repo_dir)
from scripts.notebook_ui import celebrate, show_gpu_status, show_leaderboard

# Show GPU status
gpu_name = 'Colab GPU'
try:
    r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,utilization.gpu',
                        '--format=csv,noheader'], capture_output=True, text=True)
    if r.returncode == 0:
        parts = [x.strip() for x in r.stdout.strip().split(',')]
        gpu_name = parts[0]
        show_gpu_status(parts[0], parts[1], parts[2])
except: pass

# Find ALL log files (not just colab-tagged)
import re
logs = []
for pattern in ['scripts/experiments/*/results/*.log', 'data/**/*.log']:
    for f in glob.glob(os.path.join(repo_dir, pattern), recursive=True):
        # Check if it has results (completed)
        try:
            content = open(f).read()
            if 'RESULTS' in content and ('Density:' in content or 'Uncovered:' in content):
                logs.append(f)
        except: pass

# Sort by modification time, show most recent
logs.sort(key=os.path.getmtime, reverse=True)

if logs:
    print(f'Found {len(logs)} completed experiment(s)\n')
    for log_file in logs[:3]:  # celebrate top 3
        with open(log_file) as f:
            content = f.read()
        stats = {}
        m = re.search(r'Density:\s*([\d.]+)%', content)
        if m: stats['Density'] = f'{m.group(1)}%'
        m = re.search(r'Uncovered:\s*(\d+)', content)
        if m: stats['Uncovered'] = m.group(1)
        m = re.search(r'Range: d = 1 to (\d+)', content)
        rng = f'10^{len(m.group(1))-1}' if m else '?'
        if m: stats['Range'] = rng
        m = re.search(r'Time:\s*([\d.]+)s', content)
        if m: stats['Time'] = f'{float(m.group(1)):.1f}s'
        m = re.search(r'Digits:\s*\{([^}]+)\}', content)
        digits = m.group(1) if m else '?'

        if stats:
            celebrate(
                f'Zaremba Density A={{{digits}}}',
                f'Verified to {rng} — {os.path.basename(log_file)}',
                stats,
                gpu_name=gpu_name
            )

    # Leaderboard
    show_leaderboard([
        {'name': '8×B200 Cluster', 'value': '10^12', 'unit': 'range'},
        {'name': '8×B200 Cluster', 'value': '10^11', 'unit': 'range'},
        {'name': f'Your {gpu_name}', 'value': rng, 'unit': 'range', 'highlight': True},
    ])
else:
    print('No completed experiments found.')
    print(f'Working directory: {os.getcwd()}')
    print(f'Looking in: {repo_dir}/scripts/experiments/*/results/*.log')
    all_logs = glob.glob(os.path.join(repo_dir, 'scripts/experiments/*/results/*.log'))
    print(f'All log files found: {len(all_logs)}')
    for f in all_logs[:5]:
        print(f'  {f}')


In [ ]:
# === Step 10: Go Deeper (you choose!) ===
# Now that you've seen quick wins, try a REAL computation.
# These take longer but produce data that doesn't exist yet.

# Pick ONE and uncomment it:

# --- Zaremba A={1,2,8} at 10^9 (~2 min) ---
# New digit set! Nobody has checked {1,2,8} at this range.
# !./zaremba_density_gpu 1000000000 1,2,8 | tee scripts/experiments/zaremba-density/results/gpu_A128_1e9_colab.log

# --- Zaremba A={1,2,9} at 10^9 (~2 min) ---
# Even more unexplored territory.
# !./zaremba_density_gpu 1000000000 1,2,9 | tee scripts/experiments/zaremba-density/results/gpu_A129_1e9_colab.log

# --- Zaremba A={1,2,3} at 10^9 (~5 min) ---
# Confirm the famous 27 exceptions at a bigger range.
# !./zaremba_density_gpu 1000000000 1,2,3 | tee scripts/experiments/zaremba-density/results/gpu_A123_1e9_colab.log

# --- Zaremba A={1,2,8} at 10^10 (~30 min, 1.25 GB VRAM) ---
# Serious computation. Extends the {1,2,k} hierarchy.
# !./zaremba_density_gpu 10000000000 1,2,8 | tee scripts/experiments/zaremba-density/results/gpu_A128_1e10_colab.log

# --- Kronecker S_25 (~2 min char table + ~30 sec GPU) ---
# Bigger symmetric group. More triples than anyone has published.
# !python3 scripts/experiments/kronecker-coefficients/char_table.py 25
# !./kronecker_gpu 25 | tee scripts/experiments/kronecker-coefficients/results/kronecker_n25_colab.log

# --- Ramanujan Machine degree 4 (~5 min) ---
# Search for new continued fraction formulas for mathematical constants.
# If you find one, it's a genuine mathematical discovery.
# !./ramanujan_gpu 4 3 | tee scripts/experiments/ramanujan-machine/results/ramanujan_deg4_r3_colab.log


In [ ]:
# === Step 12: Harvest your results ===
# The agent picks up the log file and analyzes it
!python3 scripts/research_agent.py --once --phase harvest


In [ ]:
# === Step 13: Download your results to submit as a PR ===
# Your computation extended the frontier. Download the log files
# and submit them as a PR to cahlen/idontknow.

import glob
logs = glob.glob('scripts/experiments/*/results/*colab*.log') + glob.glob('data/**/colab*.log', recursive=True)
reviews = glob.glob('docs/verifications/*review*.json')

print(f'Your results: {len(logs)} experiment log(s), {len(reviews)} review(s)\n')
for f in logs + reviews[-5:]:
    print(f'  {f}')

# In Colab, use files.download() to get them
try:
    from google.colab import files
    print('\nClick to download:')
    for f in logs:
        files.download(f)
except:
    print('\nFiles are in the paths above. Download and submit a PR to:')
    print('  https://github.com/cahlen/idontknow')


In [ ]:
# === Step 10: Analyze YOUR results ===
# Let's have an AI look at what YOU just computed.

import os, sys, glob, re
sys.path.insert(0, '.')

repo_dir = '/content/idontknow' if os.path.exists('/content/idontknow') else '.'
os.chdir(repo_dir)

# Find your most recent experiment log
logs = []
for pattern in ['scripts/experiments/*/results/*.log', 'data/**/*.log']:
    for f in glob.glob(pattern, recursive=True):
        try:
            content = open(f).read()
            if 'RESULTS' in content:
                logs.append((f, content, os.path.getmtime(f)))
logs.sort(key=lambda x: x[2], reverse=True)

if not logs:
    print('No experiment results found. Run Step 8 first.')
else:
    log_file, content, _ = logs[0]
    print(f'Analyzing: {log_file}\n')
    # Show the results section
    if 'RESULTS' in content:
        print(content[content.index('RESULTS'):])

    # Call LLM directly (plain text, not JSON)
    import httpx
    prompt = (
        'A user just ran this GPU experiment on Google Colab as part of '
        'bigcompute.science open math research. Analyze the results in 3-4 sentences: '
        '(1) what they computed, (2) whether it matches known math, '
        '(3) whether this is new data. Be encouraging but honest. '
        'Respond with PLAIN TEXT only, no JSON, no markdown fences.\n\n'
        f'Results:\n{content[-800:]}'
    )

    analysis = None
    gemini_key = os.environ.get('GEMINI_API_KEY', os.environ.get('GOOGLE_API_KEY', ''))
    openai_key = os.environ.get('OPENAI_API_KEY', '')

    if gemini_key:
        try:
            resp = httpx.post(
                'https://generativelanguage.googleapis.com/v1beta/openai/chat/completions',
                headers={'Authorization': f'Bearer {gemini_key}', 'Content-Type': 'application/json'},
                json={'model': 'gemini-3-flash', 'messages': [{'role': 'user', 'content': prompt}],
                      'max_completion_tokens': 500},
                timeout=30.0,
            )
            if resp.status_code == 200:
                analysis = resp.json()['choices'][0]['message']['content'].strip()
        except Exception as e:
            print(f'Gemini error: {e}')
    elif openai_key:
        try:
            resp = httpx.post(
                'https://api.openai.com/v1/chat/completions',
                headers={'Authorization': f'Bearer {openai_key}', 'Content-Type': 'application/json'},
                json={'model': 'gpt-4.1-mini', 'messages': [{'role': 'user', 'content': prompt}],
                      'max_completion_tokens': 500},
                timeout=30.0,
            )
            if resp.status_code == 200:
                analysis = resp.json()['choices'][0]['message']['content'].strip()
        except Exception as e:
            print(f'OpenAI error: {e}')

    if analysis:
        # Strip any JSON/markdown the model wraps around the answer
        if '```' in analysis:
            analysis = analysis.split('```')[1] if '```' in analysis else analysis
            if analysis.startswith('json'): analysis = analysis[4:]
            analysis = analysis.strip()
        if analysis.lstrip().startswith('{'):
            import json as _json
            try:
                parsed = _json.loads(analysis)
                # Recursively extract all string values
                def extract_text(obj):
                    if isinstance(obj, str): return obj
                    if isinstance(obj, dict): return ' '.join(extract_text(v) for v in obj.values())
                    if isinstance(obj, list): return ' '.join(extract_text(v) for v in obj)
                    return str(obj)
                analysis = extract_text(parsed)
            except: pass
        if analysis.startswith('```'):
            analysis = analysis.split('\n', 1)[-1].rsplit('```', 1)[0].strip()

        from IPython.display import HTML, display
        display(HTML(f'''
        <div style="background:#0b0d10;border-left:4px solid #e8c47a;padding:1rem 1.5rem;margin:1rem 0;max-width:600px;font-family:Georgia,serif;">
          <div style="font-size:0.6rem;text-transform:uppercase;letter-spacing:0.12em;color:#e8c47a;font-weight:bold;margin-bottom:0.5rem;">AI Analysis of Your Computation</div>
          <p style="font-size:0.9rem;color:#e8e6e3;line-height:1.6;margin:0;">{analysis}</p>
        </div>
        '''))
    else:
        print('Could not analyze — check your API key in Step 3.')


In [ ]:
# === Review results ===
import sys, glob, json
sys.path.insert(0, '.')
from scripts.notebook_ui import show_review

# Find the most recently created review
import os
reviews = sorted(glob.glob('docs/verifications/*review*.json'),
                 key=os.path.getmtime, reverse=True)

for rf in reviews[:3]:  # show last 3 reviews
    try:
        with open(rf) as f:
            r = json.load(f)
        show_review(
            r.get('finding_slug', '?'),
            r.get('reviewer', {}).get('model', '?'),
            r.get('overall_verdict', '?'),
            r.get('certification_recommendation', '?')
        )
    except: pass


In [ ]:
# === Step 11: View all reviews across all findings ===
import json, glob, os

# Collect all review files grouped by finding
reviews_by_slug = {}
for f in sorted(glob.glob('docs/verifications/*review*.json')):
    try:
        with open(f) as fh:
            r = json.load(fh)
        slug = r.get('finding_slug', '?')
        model = r.get('reviewer', {}).get('model', '?')
        verdict = r.get('overall_verdict', '?')
        cert = r.get('certification_recommendation', '?')
        if slug not in reviews_by_slug:
            reviews_by_slug[slug] = []
        reviews_by_slug[slug].append((model, verdict, cert))
    except:
        continue

print(f'{sum(len(v) for v in reviews_by_slug.values())} reviews across {len(reviews_by_slug)} findings:\n')
for slug in sorted(reviews_by_slug.keys()):
    revs = reviews_by_slug[slug]
    models = ', '.join(set(r[0] for r in revs if r[0] != '?'))
    worst = min(revs, key=lambda r: {'REJECT':0,'REVISE_AND_RESUBMIT':1,'ACCEPT_WITH_REVISION':2,'ACCEPT':3}.get(r[1], 2))
    print(f'  {slug[:45]:45s} {len(revs)} reviews  {worst[1]:25s} [{models}]')


In [ ]:
# === Step 12: Rebuild manifest (aggregate all reviews) ===
!python3 scripts/reviews/aggregate.py
!python3 scripts/reviews/validate.py --all

In [ ]:
# === Step 13: Review ALL findings at once ===
# This will take a few minutes (15 findings x ~30s each)
# Gemini free tier: 15 RPM, so this stays within limits

if os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY'):
    key = os.environ.get('GEMINI_API_KEY', os.environ.get('GOOGLE_API_KEY', ''))
    os.environ['API_KEY'] = key
    !python3 scripts/reviews/run_review.py --all --model gemini-3-flash --provider google --api-base https://generativelanguage.googleapis.com/v1beta/openai --skip-existing
elif os.environ.get('OPENAI_API_KEY'):
    os.environ['API_KEY'] = os.environ['OPENAI_API_KEY']
    !python3 scripts/reviews/run_review.py --all --model gpt-4.1 --provider openai --skip-existing
else:
    print('Set GEMINI_API_KEY or OPENAI_API_KEY first')

## Contribute Your Reviews

**Your review adds a new perspective.** Different models catch different errors. Claude found 6 issues in Round 1. o3-pro found the prime count error. Your Gemini review might find something they all missed.

**To submit:**
1. Download the review JSON(s) from `docs/verifications/`
2. Open a PR to [cahlen/idontknow](https://github.com/cahlen/idontknow)
3. Your review will be added to the audit ledger on [bigcompute.science/verification](https://bigcompute.science/verification/)

**To run the full autonomous loop** (needs a machine with GPUs):
```bash
export GEMINI_API_KEY='AIza...'  # or OPENAI_API_KEY or ANTHROPIC_API_KEY
nohup ./scripts/run_agent.sh --loop 10m &
```

**More info:** [AGENTS.md](https://github.com/cahlen/idontknow/blob/main/AGENTS.md) · [Verification](https://bigcompute.science/verification/) · [MCP Server](https://mcp.bigcompute.science/mcp) (23 tools)